<a href="https://www.kaggle.com/code/robiulhasanjisan/cdanet-experiment-v1?scriptVersionId=318012043" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:


import os
import cv2
import timm
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import tqdm
from glob import glob

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score
)

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader

from albumentations.pytorch import ToTensorV2
import albumentations as A

warnings.filterwarnings("ignore")

In [ ]:
def seed_everything(seed=42):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)

# LOAD DATA

In [ ]:
DATASET_PATH = "/kaggle/input/datasets/superlord/citrus-diseases/Citrus-Diseases"

IMG_SIZE = 224
BATCH_SIZE = 8
EPOCHS = 25
LR = 1e-4

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("DEVICE:", DEVICE)


#

# CLASSES

In [ ]:
classes = {
    "aphids":0,
    "gummosis":1,
    "leaf_minnor":2,
    "healthy":3
}

idx_to_class = {v:k for k,v in classes.items()}

# BUILD DATAFRAME

In [ ]:
data = []

for cls in classes:

    files = glob(
        os.path.join(DATASET_PATH, cls, "*.jpg")
    )

    for f in files:

        data.append([f, classes[cls]])

df = pd.DataFrame(data, columns=["path", "label"])

print(df.shape)

# TRAIN VALID SPLIT

In [ ]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)


# AUGMENTATION

In [ ]:

train_transform = A.Compose([

    A.Resize(256,256),

    A.RandomCrop(IMG_SIZE, IMG_SIZE),

    A.HorizontalFlip(p=0.5),

    A.VerticalFlip(p=0.3),

    A.Rotate(limit=30, p=0.5),

    A.RandomBrightnessContrast(
        brightness_limit=0.3,
        contrast_limit=0.3,
        p=0.5
    ),

    A.HueSaturationValue(
        hue_shift_limit=15,
        sat_shift_limit=20,
        val_shift_limit=15,
        p=0.5
    ),

    A.GaussNoise(p=0.3),

    A.CoarseDropout(
        num_holes_range=(2,6),
        hole_height_range=(20,40),
        hole_width_range=(20,40),
        p=0.3
    ),

    A.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    ),

    ToTensorV2()
])

val_transform = A.Compose([

    A.Resize(IMG_SIZE, IMG_SIZE),

    A.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    ),

    ToTensorV2()
])


# Adaptive Illumination-Aware Citrus Dataset Pipeline

In [ ]:
class CitrusDataset(Dataset):

    def __init__(self, dataframe, transform=None):

        self.df = dataframe.reset_index(drop=True)

        self.transform = transform

    def __len__(self):

        return len(self.df)

    def adaptive_illumination(self, img):

        

        lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)

        l, a, b = cv2.split(lab)

        clahe = cv2.createCLAHE(
            clipLimit=3.0,
            tileGridSize=(8,8)
        )

        l = clahe.apply(l)

        merged = cv2.merge([l,a,b])

        img = cv2.cvtColor(
            merged,
            cv2.COLOR_LAB2RGB
        )

        return img

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        img = cv2.imread(row.path)

        img = cv2.cvtColor(
            img,
            cv2.COLOR_BGR2RGB
        )

        # Adaptive illumination
        img = self.adaptive_illumination(img)

        if self.transform:

            augmented = self.transform(image=img)

            img = augmented["image"]

        label = row.label

        return img, torch.tensor(label)


# Efficient Data Loading and Augmentation Pipeline

In [ ]:
train_dataset = CitrusDataset(
    train_df,
    transform=train_transform
)

val_dataset = CitrusDataset(
    val_df,
    transform=val_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2
)

# Class-Aware Supervised Contrastive Optimization

In [ ]:
class SupConLoss(nn.Module):

    def __init__(self, temperature=0.07):

        super().__init__()

        self.temperature = temperature

    def forward(self, features, labels):

        features = F.normalize(features, dim=1)

        similarity = torch.matmul(
            features,
            features.T
        ) / self.temperature

        labels = labels.contiguous().view(-1,1)

        mask = torch.eq(
            labels,
            labels.T
        ).float().to(features.device)

        mask.fill_diagonal_(0)

        exp_sim = torch.exp(similarity)

        log_prob = similarity - torch.log(
            exp_sim.sum(dim=1, keepdim=True)
        )

        mean_log_prob_pos = (
            mask * log_prob
        ).sum(dim=1) / (mask.sum(dim=1) + 1e-8)

        loss = -mean_log_prob_pos.mean()

        return loss

# Disease Region Enhancement Module (DREM)

In [ ]:
class DREM(nn.Module):

    def __init__(self, channels):

        super().__init__()

        self.conv = nn.Sequential(

            nn.Conv2d(
                channels,
                channels,
                3,
                padding=1
            ),

            nn.BatchNorm2d(channels),

            nn.ReLU(),

            nn.Conv2d(
                channels,
                1,
                1
            ),

            nn.Sigmoid()
        )

    def forward(self, x):

        attn = self.conv(x)

        return x * attn

# Cross-Domain Attention Fusion Module

In [ ]:

class CrossDomainAttention(nn.Module):

    def __init__(self, dim1, dim2):

        super().__init__()

        self.query = nn.Linear(dim1, 512)

        self.key = nn.Linear(dim2, 512)

        self.value = nn.Linear(dim2, 512)

        self.softmax = nn.Softmax(dim=-1)

    def forward(self, feat1, feat2):

        q = self.query(feat1)

        k = self.key(feat2)

        v = self.value(feat2)

        attn = self.softmax(
            torch.matmul(q, k.T)
        )

        fused = torch.matmul(attn, v)

        return fused


# CDANet: Cross-Domain Attention Network for Uncertainty-Aware Citrus Disease Classification

In [ ]:
class CDANet(nn.Module):

    def __init__(self, num_classes=4):

        super().__init__()

        # CNN BRANCH
        self.cnn = timm.create_model(
            "efficientnet_b3",
           pretrained=False,
            features_only=True
        )

        # TRANSFORMER BRANCH
        self.vit = timm.create_model(
            "vit_small_patch16_224",
            pretrained=False,
            num_classes=0
        )

        cnn_channels = 384

        # Disease Region Enhancement
        self.drem = DREM(cnn_channels)

        # Global pooling
        self.pool = nn.AdaptiveAvgPool2d(1)

        # Cross Attention
        self.cross_attention = CrossDomainAttention(
            cnn_channels,
            384
        )

        # Projection Head
        self.projection = nn.Sequential(

            nn.Linear(512,256),

            nn.ReLU(),

            nn.Linear(256,128)
        )

        # Uncertainty Head
        self.uncertainty_head = nn.Sequential(

            nn.Linear(512,128),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(128,1),

            nn.Softplus()
        )

        # Classifier
        self.classifier = nn.Sequential(

            nn.Linear(512,256),

            nn.ReLU(),

            nn.BatchNorm1d(256),

            nn.Dropout(0.4),

            nn.Linear(256,num_classes)
        )

    def forward(self, x):

        # CNN features
        cnn_feat = self.cnn(x)[-1]

        # Disease enhancement
        cnn_feat = self.drem(cnn_feat)

        cnn_feat = self.pool(cnn_feat)

        cnn_feat = cnn_feat.view(
            cnn_feat.size(0),
            -1
        )

        # ViT features
        vit_feat = self.vit(x)

        # Cross-domain fusion
        fused = self.cross_attention(
            cnn_feat,
            vit_feat
        )

        # Contrastive feature
        proj_feat = self.projection(fused)

        # Uncertainty
        uncertainty = self.uncertainty_head(fused)

        # Final classification
        logits = self.classifier(fused)

        return logits, proj_feat, uncertainty


In [ ]:

model = CDANet(
    num_classes=4
).to(DEVICE)


## Hybrid Loss Function: Cross-Entropy and Supervised Contrastive Learning

In [ ]:
ce_loss = nn.CrossEntropyLoss()

contrastive_loss = SupConLoss()


## Optimization Strategy: AdamW with Cosine Annealing Scheduler

In [ ]:

optimizer = optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=1e-4
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)


## Epoch-Level Training with Hybrid Loss and Uncertainty Regularization

In [ ]:
def train_epoch():

    model.train()

    total_loss = 0

    preds_all = []
    labels_all = []

    for images, labels in tqdm(train_loader):

        images = images.to(DEVICE)

        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        logits, proj_feat, uncertainty = model(images)

        cls_loss = ce_loss(
            logits,
            labels
        )

        cont_loss = contrastive_loss(
            proj_feat,
            labels
        )

        # Novel uncertainty weighting
        unc_loss = uncertainty.mean()

        loss = (
            cls_loss +
            0.3 * cont_loss +
            0.1 * unc_loss
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        preds = torch.argmax(
            logits,
            dim=1
        )

        preds_all.extend(
            preds.cpu().numpy()
        )

        labels_all.extend(
            labels.cpu().numpy()
        )

    acc = accuracy_score(
        labels_all,
        preds_all
    )

    f1 = f1_score(
        labels_all,
        preds_all,
        average="weighted"
    )

    return total_loss/len(train_loader), acc, f1

## Model Validation with Hybrid Loss and Uncertainty Estimation

In [ ]:
def validate():

    model.eval()

    total_loss = 0

    preds_all = []
    labels_all = []

    uncertainties = []

    with torch.no_grad():

        for images, labels in tqdm(val_loader):

            images = images.to(DEVICE)

            labels = labels.to(DEVICE)

            logits, proj_feat, uncertainty = model(images)

            cls_loss = ce_loss(
                logits,
                labels
            )

            cont_loss = contrastive_loss(
                proj_feat,
                labels
            )

            unc_loss = uncertainty.mean()

            loss = (
                cls_loss +
                0.3 * cont_loss +
                0.1 * unc_loss
            )

            total_loss += loss.item()

            preds = torch.argmax(
                logits,
                dim=1
            )

            preds_all.extend(
                preds.cpu().numpy()
            )

            labels_all.extend(
                labels.cpu().numpy()
            )

            uncertainties.extend(
                uncertainty.cpu().numpy()
            )

    acc = accuracy_score(
        labels_all,
        preds_all
    )

    f1 = f1_score(
        labels_all,
        preds_all,
        average="weighted"
    )

    mean_uncertainty = np.mean(
        uncertainties
    )

    return (
        total_loss/len(val_loader),
        acc,
        f1,
        mean_uncertainty,
        labels_all,
        preds_all
    )


## Experimental Framework for CDANet Training and Validation

In [ ]:

best_f1 = 0

history = {
    "train_f1":[],
    "val_f1":[]
}

print("\nSTART TRAINING...\n")

for epoch in range(EPOCHS):

    print("="*60)
    print(f"EPOCH {epoch+1}/{EPOCHS}")
    print("="*60)

    train_loss, train_acc, train_f1 = train_epoch()

    (
        val_loss,
        val_acc,
        val_f1,
        val_unc,
        y_true,
        y_pred
    ) = validate()

    scheduler.step()

    history["train_f1"].append(train_f1)
    history["val_f1"].append(val_f1)

    print(f"\nTRAIN LOSS: {train_loss:.4f}")
    print(f"TRAIN ACC : {train_acc:.4f}")
    print(f"TRAIN F1  : {train_f1:.4f}")

    print(f"\nVAL LOSS  : {val_loss:.4f}")
    print(f"VAL ACC   : {val_acc:.4f}")
    print(f"VAL F1    : {val_f1:.4f}")
    print(f"UNCERTAINTY: {val_unc:.4f}")

    if val_f1 > best_f1:

        best_f1 = val_f1

        torch.save(
            model.state_dict(),
            "best_cdanet_model.pth"
        )

        print("\nBEST MODEL SAVED!")


# CLASSIFICATION REPORT

In [ ]:

print("\nCLASSIFICATION REPORT\n")

print(classification_report(
    y_true,
    y_pred,
    target_names=list(classes.keys())
))


# Confusion Matrix Analysis for CDANet Performance Evaluation

In [ ]:
cm = confusion_matrix(
    y_true,
    y_pred
)

plt.figure(figsize=(8,6))

plt.imshow(cm)

plt.title("Confusion Matrix")

plt.colorbar()

plt.xticks(
    np.arange(len(classes)),
    classes.keys(),
    rotation=45
)

plt.yticks(
    np.arange(len(classes)),
    classes.keys()
)

for i in range(cm.shape[0]):

    for j in range(cm.shape[1]):

        plt.text(
            j,
            i,
            cm[i,j],
            ha="center",
            va="center"
        )

plt.xlabel("Predicted")
plt.ylabel("True")

plt.show()


# Training Dynamics and Performance Convergence Analysis

In [ ]:
plt.figure(figsize=(10,5))

plt.plot(
    history["train_f1"],
    label="Train F1"
)

plt.plot(
    history["val_f1"],
    label="Val F1"
)

plt.legend()

plt.title("Training Curve")

plt.show()


print("CDA-NET TRAINING COMPLETE")


print(f"\nBEST VALIDATION F1: {best_f1:.4f}")
